In [ ]:
# =========================================================
# 1) CORE DATA / NUMERICAL
# =========================================================
import pandas as pd
import numpy as np

# =========================================================
# 2) VISUALISATION
# =========================================================
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import plotly.graph_objects as go

%matplotlib inline

# =========================================================
# 3) STATISTICS / RISK
# =========================================================
from scipy import stats
from scipy.stats import skew, kurtosis, norm, t, genpareto

# =========================================================
# 4) BACKTESTING
# =========================================================
import backtrader as bt

# =========================================================
# 5) DATA SOURCES
# =========================================================
import yfinance as yf

# =========================================================
# 6) DISPLAY / REPORTING
# =========================================================
from tabulate import tabulate
from IPython.display import display

# =========================================================
# 7) SYSTEM / FILES / ENV
# =========================================================
import os
from pathlib import Path
from datetime import datetime
from IPython import get_ipython

# =========================================================
# CHECK
# =========================================================
print("All libraries are imported")


In [ ]:
CAC40 = pd.read_excel(r"C:\Users\belar\Dropbox\4_Trading perso\0_input_excel\CAC40.xlsx").sort_values("Date").reset_index(drop=True)
df_cac40 = CAC40.copy()
print(df_cac40.info())

In [ ]:
import time
import numpy as np
import pandas as pd
import vectorbt as vbt
from tqdm import tqdm

# ============================================================
# DATA
# ============================================================
df = df_cac40.copy().sort_values("Date").set_index("Date")
px = df["CAC40_Close"].astype(float).dropna()

# ============================================================
# CONFIG
# ============================================================
INIT_CASH = 100000.0
FEES = 0.005

SL_STOP = 0.01   # -1% stop loss
TP_STOP = 0.02   # +2% take profit

WF_TRAIN_YEARS = 5
WF_TEST_YEARS  = 1

short_list = np.arange(2, 51, 2)
long_list  = np.arange(20, 201, 3)

MIN_TRADES_TRAIN = 10

# ============================================================
# HELPERS
# ============================================================
def _safe_count_trades(pf: vbt.Portfolio) -> int:
    try:
        return int(pf.trades.count())
    except Exception:
        try:
            return int(len(pf.trades.records))
        except Exception:
            return 0

def eval_ma_longshort(series: pd.Series, s: int, l: int) -> dict:
    if series.empty or len(series) < (l + 5):
        return {"end_value": np.nan, "sharpe": np.nan, "max_dd_pct": np.nan, "trades": 0}

    fast = vbt.MA.run(series, window=int(s)).ma
    slow = vbt.MA.run(series, window=int(l)).ma

    long_regime  = (fast > slow)
    short_regime = (fast < slow)

    prev_long  = long_regime.shift(1).fillna(False)
    prev_short = short_regime.shift(1).fillna(False)

    long_entries  = long_regime & (~prev_long)
    long_exits    = prev_long & (~long_regime)

    short_entries = short_regime & (~prev_short)
    short_exits   = prev_short & (~short_regime)

    conflict = long_entries & short_entries
    if conflict.any():
        long_entries = long_entries & (~conflict)
        short_entries = short_entries & (~conflict)

    pf = vbt.Portfolio.from_signals(
        series,
        entries=long_entries,
        exits=long_exits,
        short_entries=short_entries,
        short_exits=short_exits,
        init_cash=INIT_CASH,
        fees=FEES,
        sl_stop=SL_STOP,
        tp_stop=TP_STOP,
        direction="both",
        freq="D"
    )

    return {
        "end_value": float(pf.value().iloc[-1]),
        "sharpe": float(pf.sharpe_ratio()),
        "max_dd_pct": abs(float(pf.max_drawdown()) * 100),
        "trades": _safe_count_trades(pf)
    }

# ============================================================
# TIMER + PROGRESS SETUP
# ============================================================
start_time = time.time()

years = sorted(px.index.year.unique())
total_wf_windows = len(years) - (WF_TRAIN_YEARS + WF_TEST_YEARS) + 1
total_param_tests = max(total_wf_windows, 0) * len(short_list) * len(long_list)

print(f"Total WF windows: {total_wf_windows}")
print(f"Total param tests: {total_param_tests:,}")

wf_rows = []

param_bar = tqdm(total=total_param_tests, desc="Optimisation MA", unit="tests")

# ============================================================
# WALK-FORWARD
# ============================================================
for i in range(0, total_wf_windows):
    train_y0 = years[i]
    train_y1 = years[i + WF_TRAIN_YEARS - 1]
    test_y0  = years[i + WF_TRAIN_YEARS]
    test_y1  = years[i + WF_TRAIN_YEARS + WF_TEST_YEARS - 1]

    train = px.loc[f"{train_y0}-01-01":f"{train_y1}-12-31"]
    test  = px.loc[f"{test_y0}-01-01":f"{test_y1}-12-31"]

    best = None
    best_score = -1e18

    for s in short_list:
        for l in long_list:
            param_bar.update(1)

            if s >= l:
                continue

            r = eval_ma_longshort(train, s, l)
            if not np.isfinite(r["end_value"]):
                continue
            if r["trades"] < MIN_TRADES_TRAIN:
                continue

            score = r["sharpe"] - 0.01 * r["max_dd_pct"]

            if score > best_score:
                best_score = score
                best = {"short": int(s), "long": int(l), **r}

    if best is None:
        wf_rows.append({
            "train_period": f"{train_y0}-{train_y1}",
            "test_period": f"{test_y0}-{test_y1}",
            "best_short": None,
            "best_long": None,
            "test_end_value": INIT_CASH,
            "test_sharpe": np.nan,
            "test_max_dd_pct": 0.0,
            "test_trades": 0,
            "note": "no valid params (too few trades)"
        })
        continue

    test_r = eval_ma_longshort(test, best["short"], best["long"])

    wf_rows.append({
        "train_period": f"{train_y0}-{train_y1}",
        "test_period": f"{test_y0}-{test_y1}",
        "best_short": best["short"],
        "best_long": best["long"],
        "test_end_value": round(float(test_r["end_value"]), 2),
        "test_sharpe": float(test_r["sharpe"]),
        "test_max_dd_pct": round(float(test_r["max_dd_pct"]), 2),
        "test_trades": int(test_r["trades"]),
        "note": ""
    })

param_bar.close()

# ============================================================
# RESULTS
# ============================================================
wf = pd.DataFrame(wf_rows)
display(wf.head(15))

print("\nWF median:")
print(wf[["test_end_value", "test_sharpe", "test_max_dd_pct", "test_trades"]].median(numeric_only=True))

print("\nBest params frequency (stability):")
print(
    wf.dropna(subset=["best_short", "best_long"])
      .value_counts(["best_short", "best_long"])
      .sort_values(ascending=False)
      .head(10)
)

# ============================================================
# TIMER END
# ============================================================
elapsed = time.time() - start_time
print("\nExecution time:")
print(f"{elapsed:.2f} seconds")
print(f"{elapsed/60:.2f} minutes")


In [ ]:
import time
import numpy as np
import pandas as pd
import vectorbt as vbt
from tqdm import tqdm

# ============================================================
# 0) DATA
# ============================================================
df = df_cac40.copy().sort_values("Date").set_index("Date")
px = df["CAC40_Close"].astype(float).dropna()

# ============================================================
# 1) CONFIG
# ============================================================
INIT_CASH = 100000.0
FEES = 0.005

WF_TRAIN_YEARS = 5
WF_TEST_YEARS  = 1

short_list = np.arange(2, 101, 2)
long_list  = np.arange(20, 201, 3)

MIN_TRADES_TRAIN = 5
DD_PENALTY = 0.01

# ============================================================
# 2) BUILD ALL MA PAIRS
# ============================================================
pairs = [(int(s), int(l)) for s in short_list for l in long_list if s < l]
pairs_index = pd.MultiIndex.from_tuples(pairs, names=["short", "long"])

short_arr = np.array([p[0] for p in pairs], dtype=int)
long_arr  = np.array([p[1] for p in pairs], dtype=int)

all_windows = np.unique(np.r_[short_list, long_list]).astype(int)

# ============================================================
# 3) VECTORISED TRAIN EVAL (LONG + SHORT)
# ============================================================
def eval_all_pairs_train(series: pd.Series) -> pd.DataFrame:

    if series.empty:
        return pd.DataFrame(index=pairs_index)

    # ---- MA calculées une seule fois
    ma_all = vbt.MA.run(series, window=all_windows).ma
    if isinstance(ma_all.columns, pd.MultiIndex):
        ma_all.columns = ma_all.columns.get_level_values(-1).astype(int)
    else:
        ma_all.columns = ma_all.columns.astype(int)

    fast = ma_all.loc[:, short_arr].to_numpy()
    slow = ma_all.loc[:, long_arr].to_numpy()

    # ---- Régimes
    long_reg  = fast > slow
    short_reg = fast < slow

    prev_long  = np.vstack([np.zeros((1, long_reg.shape[1]), dtype=bool),  long_reg[:-1]])
    prev_short = np.vstack([np.zeros((1, short_reg.shape[1]), dtype=bool), short_reg[:-1]])

    long_entries  = long_reg  & (~prev_long)
    long_exits    = prev_long & (~long_reg)

    short_entries = short_reg & (~prev_short)
    short_exits   = prev_short & (~short_reg)

    idx = series.index

    long_entries  = pd.DataFrame(long_entries,  index=idx, columns=pairs_index)
    long_exits    = pd.DataFrame(long_exits,    index=idx, columns=pairs_index)
    short_entries = pd.DataFrame(short_entries, index=idx, columns=pairs_index)
    short_exits   = pd.DataFrame(short_exits,   index=idx, columns=pairs_index)

    # ---- Portfolio LONG/SHORT
    pf = vbt.Portfolio.from_signals(
        close=series,
        entries=long_entries,
        exits=long_exits,
        short_entries=short_entries,
        short_exits=short_exits,
        init_cash=INIT_CASH,
        fees=FEES,
        freq="D"
    )

    # ---- Trades count robuste
    try:
        trades = pf.trades.count()
    except Exception:
        trades = pd.Series([len(pf.trades.records)] * len(pairs_index), index=pairs_index)

    metrics = pd.DataFrame({
        "end_value": pf.value().iloc[-1],
        "sharpe": pf.sharpe_ratio(),
        "max_dd_pct": pf.max_drawdown().abs() * 100,
        "trades": trades
    }, index=pairs_index)

    return metrics

# ============================================================
# 4) TEST EVAL (1 pair seulement)
# ============================================================
def eval_one_pair_test(series: pd.Series, s: int, l: int) -> dict:

    if series.empty or len(series) < (l + 5):
        return {"end_value": np.nan, "sharpe": np.nan, "max_dd_pct": np.nan, "trades": 0}

    fast = vbt.MA.run(series, window=int(s)).ma
    slow = vbt.MA.run(series, window=int(l)).ma

    long_reg  = (fast > slow).astype(bool)
    short_reg = (fast < slow).astype(bool)

    prev_long  = long_reg.shift(1, fill_value=False)
    prev_short = short_reg.shift(1, fill_value=False)

    long_entries  = long_reg  & (~prev_long)
    long_exits    = prev_long & (~long_reg)

    short_entries = short_reg & (~prev_short)
    short_exits   = prev_short & (~short_reg)

    pf = vbt.Portfolio.from_signals(
        close=series,
        entries=long_entries,
        exits=long_exits,
        short_entries=short_entries,
        short_exits=short_exits,
        init_cash=INIT_CASH,
        fees=FEES,
        freq="D"
    )

    try:
        n_trades = int(pf.trades.count())
    except Exception:
        try:
            n_trades = int(len(pf.trades.records))
        except Exception:
            n_trades = 0

    return {
        "end_value": float(pf.value().iloc[-1]),
        "sharpe": float(pf.sharpe_ratio()),
        "max_dd_pct": abs(float(pf.max_drawdown()) * 100),
        "trades": n_trades
    }

# ============================================================
# 5) WALK-FORWARD
# ============================================================
start_time = time.time()

years = sorted(px.index.year.unique())
total_wf_windows = len(years) - (WF_TRAIN_YEARS + WF_TEST_YEARS) + 1

wf_rows = []

for i in tqdm(range(total_wf_windows), desc="Walk-forward", unit="window"):

    train_y0 = years[i]
    train_y1 = years[i + WF_TRAIN_YEARS - 1]
    test_y0  = years[i + WF_TRAIN_YEARS]
    test_y1  = years[i + WF_TRAIN_YEARS + WF_TEST_YEARS - 1]

    train = px.loc[f"{train_y0}-01-01":f"{train_y1}-12-31"]
    test  = px.loc[f"{test_y0}-01-01":f"{test_y1}-12-31"]

    m = eval_all_pairs_train(train)

    m = m[m["trades"] >= MIN_TRADES_TRAIN].copy()

    if m.empty:
        wf_rows.append({
            "train_period": f"{train_y0}-{train_y1}",
            "test_period": f"{test_y0}-{test_y1}",
            "best_short": None,
            "best_long": None,
            "test_end_value": INIT_CASH,
            "test_sharpe": np.nan,
            "test_max_dd_pct": 0.0,
            "test_trades": 0,
            "note": "no valid MA"
        })
        continue

    score = m["sharpe"].fillna(-1e9) - DD_PENALTY * m["max_dd_pct"].fillna(1e9)

    best_col = score.idxmax()
    best_short, best_long = int(best_col[0]), int(best_col[1])

    test_r = eval_one_pair_test(test, best_short, best_long)

    wf_rows.append({
        "train_period": f"{train_y0}-{train_y1}",
        "test_period": f"{test_y0}-{test_y1}",
        "best_short": best_short,
        "best_long": best_long,
        "test_end_value": round(test_r["end_value"], 2),
        "test_sharpe": test_r["sharpe"],
        "test_max_dd_pct": round(test_r["max_dd_pct"], 2),
        "test_trades": test_r["trades"],
        "note": ""
    })

wf = pd.DataFrame(wf_rows)
display(wf)

print("\nWF median:")
print(wf[["test_end_value","test_sharpe","test_max_dd_pct","test_trades"]].median(numeric_only=True))

print("\nBest MA frequency (stability):")
print(
    wf.dropna(subset=["best_short","best_long"])
      .value_counts(["best_short","best_long"])
      .sort_values(ascending=False)
      .head(15)
)

elapsed = time.time() - start_time
print(f"\nExecution time: {elapsed:.2f}s ({elapsed/60:.2f} min)")


In [ ]:
import time
import gc
import numpy as np
import pandas as pd
import vectorbt as vbt
from tqdm import tqdm

# ============================================================
# DATA
# ============================================================
df = df_cac40.copy().sort_values("Date").set_index("Date")
px = df["CAC40_Close"].astype(float).dropna()

# ============================================================
# CONFIG WF
# ============================================================
INIT_CASH = 100000.0
FEES = 0.005

WF_TRAIN_YEARS = 5
WF_TEST_YEARS  = 1
MIN_TRADES_TRAIN = 5   # mets 1-3 si tu veux garder des fenêtres avec peu de trades

# ============================================================
# 1) MA: on fixe une shortlist (issue de ton run "MA only")
#    Mets ce que tu veux, ex: le plus stable chez toi: (84,128)
# ============================================================
MA_CANDIDATES = [(84, 128), (2, 194), (90, 149)]  # ajuste si besoin

# ============================================================
# 2) Grille SL/TP (petite au début, sinon ça explose en temps)
# ============================================================
sl_list = np.arange(0.005, 0.051, 0.005)   # 0.5% .. 5% step 0.5%
tp_list = np.arange(0.005, 0.101, 0.005)   # 0.5% .. 10% step 0.5%

# ============================================================
# HELPERS
# ============================================================
def safe_trade_count(pf) -> int:
    try:
        return int(pf.trades.count())
    except Exception:
        try:
            return int(len(pf.trades.records))
        except Exception:
            return 0

def eval_one_pair_sl_tp(series: pd.Series, s: int, l: int, sl: float, tp: float) -> dict:
    """Eval LONG/SHORT (fast>slow = long, fast<slow = short) + SL/TP.
       Attention: short_regime = fast < slow => on est "short" en portefeuille.
    """
    if series.empty or len(series) < (l + 5):
        return {"end_value": np.nan, "sharpe": np.nan, "max_dd_pct": np.nan, "trades": 0}

    fast = vbt.MA.run(series, window=int(s)).ma
    slow = vbt.MA.run(series, window=int(l)).ma

    long_reg  = (fast > slow)
    short_reg = (fast < slow)

    prev_long  = long_reg.shift(1, fill_value=False)
    prev_short = short_reg.shift(1, fill_value=False)

    long_entries  = long_reg & (~prev_long)
    long_exits    = prev_long & (~long_reg)

    short_entries = short_reg & (~prev_short)
    short_exits   = prev_short & (~short_reg)

    pf = vbt.Portfolio.from_signals(
        close=series,
        entries=long_entries,
        exits=long_exits,
        short_entries=short_entries,
        short_exits=short_exits,
        init_cash=INIT_CASH,
        fees=FEES,
        sl_stop=float(sl),
        tp_stop=float(tp),
        freq="D"
    )

    return {
        "end_value": float(pf.value().iloc[-1]),
        "sharpe": float(pf.sharpe_ratio()),
        "max_dd_pct": abs(float(pf.max_drawdown()) * 100),
        "trades": safe_trade_count(pf)
    }

# ============================================================
# WALK-FORWARD: MA fixes -> optimiser SL/TP sur TRAIN -> appliquer sur TEST
# ============================================================
start_time = time.time()

years = sorted(px.index.year.unique())
total_wf = len(years) - (WF_TRAIN_YEARS + WF_TEST_YEARS) + 1
if total_wf <= 0:
    raise ValueError("Pas assez d'années pour WF_TRAIN_YEARS / WF_TEST_YEARS.")

wf_rows = []

# Progress total (fenêtres * MAs * SL * TP)
total_tests = total_wf * len(MA_CANDIDATES) * len(sl_list) * len(tp_list)
pbar = tqdm(total=total_tests, desc="WF SL/TP on fixed MAs", unit="test")

for i in range(total_wf):
    train_y0 = years[i]
    train_y1 = years[i + WF_TRAIN_YEARS - 1]
    test_y0  = years[i + WF_TRAIN_YEARS]
    test_y1  = years[i + WF_TRAIN_YEARS + WF_TEST_YEARS - 1]

    train = px.loc[f"{train_y0}-01-01":f"{train_y1}-12-31"]
    test  = px.loc[f"{test_y0}-01-01":f"{test_y1}-12-31"]

    best = None
    best_score = -1e18

    # --- Optim SL/TP sur TRAIN pour chaque MA candidate ---
    for (s, l) in MA_CANDIDATES:
        for sl in sl_list:
            for tp in tp_list:
                pbar.update(1)

                r = eval_one_pair_sl_tp(train, s, l, sl, tp)
                if not np.isfinite(r["end_value"]):
                    continue
                if r["trades"] < MIN_TRADES_TRAIN:
                    continue

                # score simple: Sharpe - pénalité DD
                score = r["sharpe"] - 0.01 * r["max_dd_pct"]

                if score > best_score:
                    best_score = score
                    best = {"short": int(s), "long": int(l), "sl": float(sl), "tp": float(tp), **r}

    # --- Appliquer best sur TEST ---
    if best is None:
        wf_rows.append({
            "train_period": f"{train_y0}-{train_y1}",
            "test_period": f"{test_y0}-{test_y1}",
            "best_short": None,
            "best_long": None,
            "best_sl": None,
            "best_tp": None,
            "test_end_value": INIT_CASH,
            "test_sharpe": np.nan,
            "test_max_dd_pct": 0.0,
            "test_trades": 0,
            "note": "no valid params (too few trades)"
        })
        continue

    test_r = eval_one_pair_sl_tp(test, best["short"], best["long"], best["sl"], best["tp"])

    wf_rows.append({
        "train_period": f"{train_y0}-{train_y1}",
        "test_period": f"{test_y0}-{test_y1}",
        "best_short": best["short"],
        "best_long": best["long"],
        "best_sl": round(best["sl"], 6),
        "best_tp": round(best["tp"], 6),
        "test_end_value": round(float(test_r["end_value"]), 2),
        "test_sharpe": float(test_r["sharpe"]),
        "test_max_dd_pct": round(float(test_r["max_dd_pct"]), 2),
        "test_trades": int(test_r["trades"]),
        "note": ""
    })

    # libérer un peu la mémoire entre fenêtres
    gc.collect()

pbar.close()

wf_sl_tp = pd.DataFrame(wf_rows)
display(wf_sl_tp)

print("\nWF median:")
print(wf_sl_tp[["test_end_value", "test_sharpe", "test_max_dd_pct", "test_trades"]].median(numeric_only=True))

print("\nBest (MA, SL, TP) frequency (stability):")
print(
    wf_sl_tp.dropna(subset=["best_short", "best_long", "best_sl", "best_tp"])
            .value_counts(["best_short", "best_long", "best_sl", "best_tp"])
            .sort_values(ascending=False)
            .head(15)
)

elapsed = time.time() - start_time
print(f"\nExecution time: {elapsed:.2f}s ({elapsed/60:.2f} min)")
